
# Buy Fee Model V9.1 — Lightweight Decision Engine

## Why this version

The prior V9 notebook used multiple ML pipelines, Random Forests, one-hot encoding, and large in-memory transformations. That can be unstable on a normal laptop.

V9.1 is intentionally lightweight.

It uses the already-cleaned and matched V8.2 Difference-in-Differences results directly to answer the business question:

> For +$10, +$15, +$20, +$25, and +$30 Buy Fee increases, what historical RPU gain and downside risk do we observe after adjusting against matched controls?

## Outputs

For each scenario:

- Expected RPU gain
- 95% grouped-bootstrap confidence interval
- P(Conversion decline > 1 pp)
- P(Conversion decline > 2 pp)
- P(Conversion decline > 5 pp)
- P(Volume decline > 3%)
- P(Volume decline > 10%)
- Volume risk band
- Conversion risk band
- Historical event count
- Unique locations
- Corporate change dates
- Evidence strength
- Scenario readiness

## Important

This version does **not** force a predictive ML model where V8 showed weak generalized signal.

It uses the strongest evidence we have:
- matched controls,
- same-calendar-period comparisons,
- Difference-in-Differences,
- grouped bootstrap by `Corporate_Change_ID`.


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================

DATA_FOLDER = Path(
    r"C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion"
)

V82_DIR = DATA_FOLDER / "buy_fee_model_v8_2_outputs"

DID_FILE = V82_DIR / "04_DID_EVENT_LEVEL_RESULTS.csv"

OUTPUT_DIR = DATA_FOLDER / "buy_fee_model_v9_1_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SCENARIOS = [10, 15, 20, 25, 30]

BOOTSTRAP_ITERATIONS = 500
RANDOM_STATE = 42

# Risk thresholds
CONVERSION_LOW_RISK = 0.20
CONVERSION_HIGH_RISK = 0.40

VOLUME_LOW_RISK = 0.20
VOLUME_HIGH_RISK = 0.40

# Evidence thresholds
HIGH_MIN_EVENTS = 100
HIGH_MIN_DATES = 10
HIGH_MIN_LOCS = 10

MEDIUM_MIN_EVENTS = 30
MEDIUM_MIN_DATES = 5
MEDIUM_MIN_LOCS = 5

print("=" * 80)
print("V9.1 INPUT CHECK")
print("=" * 80)

print("DiD file:")
print(DID_FILE)

print("\nExists:")
print(DID_FILE.exists())

if not DID_FILE.exists():
    raise FileNotFoundError(
        f"Missing V8.2 DiD file:\n{DID_FILE}"
    )


## 1. Load the matched-control DiD results

In [ ]:

did = pd.read_csv(
    DID_FILE,
    low_memory=True
)

print("Rows:", f"{len(did):,}")
print("Columns:", len(did.columns))

required = [
    "Segment_Change_ID",
    "Corporate_Change_ID",
    "BUYFEE_CHANGE_AMOUNT",
    "Treated_Location",
    "DiD_Volume_Change_Pct",
    "DiD_Conversion_Change_pp",
    "DiD_RPU_Change_Dollar",
]

missing = [
    c for c in required
    if c not in did.columns
]

if missing:
    raise ValueError(
        "Missing required columns: "
        + ", ".join(missing)
    )

numeric_cols = [
    "BUYFEE_CHANGE_AMOUNT",
    "DiD_Volume_Change_Pct",
    "DiD_Conversion_Change_pp",
    "DiD_RPU_Change_Dollar",
]

for c in numeric_cols:
    did[c] = pd.to_numeric(
        did[c],
        errors="coerce"
    )

did["Corporate_Change_ID"] = (
    did["Corporate_Change_ID"]
    .astype("string")
    .fillna("")
)

did["Treated_Location"] = (
    did["Treated_Location"]
    .astype("string")
    .fillna("")
)

print("\nUnique locations:", did["Treated_Location"].nunique())
print("Corporate change dates:", did["Corporate_Change_ID"].nunique())

display(did.head())


## 2. Helper functions

In [ ]:

def grouped_bootstrap_ci(
    df,
    value_col,
    n_iter=500,
    random_state=42,
    stat="median"
):
    """
    Resample Corporate_Change_ID groups rather than individual rows.
    This preserves the fact that many segment rows can come from the
    same company pricing action.
    """

    d = df[
        ["Corporate_Change_ID", value_col]
    ].dropna().copy()

    if d.empty:
        return np.nan, np.nan, np.nan

    groups = (
        d["Corporate_Change_ID"]
        .dropna()
        .unique()
    )

    if len(groups) == 1:
        vals = d[value_col]

        point = (
            vals.median()
            if stat == "median"
            else vals.mean()
        )

        return point, np.nan, np.nan

    rng = np.random.default_rng(
        random_state
    )

    boot_stats = []

    for _ in range(n_iter):

        sampled_groups = rng.choice(
            groups,
            size=len(groups),
            replace=True
        )

        pieces = []

        for g in sampled_groups:
            pieces.append(
                d[
                    d["Corporate_Change_ID"].eq(g)
                ]
            )

        sample = pd.concat(
            pieces,
            ignore_index=True
        )

        vals = sample[value_col]

        if stat == "median":
            boot_stats.append(
                vals.median()
            )
        else:
            boot_stats.append(
                vals.mean()
            )

    original = d[value_col]

    point = (
        original.median()
        if stat == "median"
        else original.mean()
    )

    low = np.nanpercentile(
        boot_stats,
        2.5
    )

    high = np.nanpercentile(
        boot_stats,
        97.5
    )

    return point, low, high


def evidence_strength(
    n_events,
    n_dates,
    n_locations
):
    if (
        n_events >= HIGH_MIN_EVENTS
        and n_dates >= HIGH_MIN_DATES
        and n_locations >= HIGH_MIN_LOCS
    ):
        return "High"

    if (
        n_events >= MEDIUM_MIN_EVENTS
        and n_dates >= MEDIUM_MIN_DATES
        and n_locations >= MEDIUM_MIN_LOCS
    ):
        return "Medium"

    return "Limited"


def conversion_risk_band(p_decline_2pp):
    if pd.isna(p_decline_2pp):
        return "Unknown"

    if p_decline_2pp < CONVERSION_LOW_RISK:
        return "Low"

    if p_decline_2pp < CONVERSION_HIGH_RISK:
        return "Moderate"

    return "High"


def volume_risk_band(
    p_decline_3pct,
    p_decline_10pct
):
    if pd.isna(p_decline_3pct):
        return "Unknown"

    if (
        p_decline_10pct >= VOLUME_HIGH_RISK
        or p_decline_3pct >= 0.60
    ):
        return "High"

    if (
        p_decline_3pct >= VOLUME_LOW_RISK
        or p_decline_10pct >= VOLUME_LOW_RISK
    ):
        return "Moderate"

    return "Low"


def scenario_readiness(
    evidence,
    rpu_ci_low,
    rpu_ci_high
):
    """
    Evidence drives readiness.
    The CI is shown separately and is not hidden.
    """

    if evidence == "High":
        return "GO"

    if evidence == "Medium":
        return "GO WITH CAUTION"

    return "RESEARCH ONLY"


## 3. Build scenario results

In [ ]:

scenario_rows = []

for fee in SCENARIOS:

    d = did[
        did["BUYFEE_CHANGE_AMOUNT"].between(
            fee - 2.5,
            fee + 2.5
        )
    ].copy()

    n_events = len(d)
    n_locations = d["Treated_Location"].nunique()
    n_dates = d["Corporate_Change_ID"].nunique()

    # --------------------------------------------------------
    # RPU
    # --------------------------------------------------------

    rpu_point, rpu_low, rpu_high = grouped_bootstrap_ci(
        d,
        "DiD_RPU_Change_Dollar",
        n_iter=BOOTSTRAP_ITERATIONS,
        random_state=RANDOM_STATE + fee,
        stat="median"
    )

    # --------------------------------------------------------
    # Conversion downside probabilities
    # --------------------------------------------------------

    conv = pd.to_numeric(
        d["DiD_Conversion_Change_pp"],
        errors="coerce"
    ).dropna()

    if len(conv):
        p_conv_1 = (
            conv <= -1
        ).mean()

        p_conv_2 = (
            conv <= -2
        ).mean()

        p_conv_5 = (
            conv <= -5
        ).mean()

        median_conv = conv.median()

    else:
        p_conv_1 = np.nan
        p_conv_2 = np.nan
        p_conv_5 = np.nan
        median_conv = np.nan

    # --------------------------------------------------------
    # Volume downside probabilities
    # --------------------------------------------------------

    vol = pd.to_numeric(
        d["DiD_Volume_Change_Pct"],
        errors="coerce"
    ).dropna()

    if len(vol):
        p_vol_3 = (
            vol <= -0.03
        ).mean()

        p_vol_10 = (
            vol <= -0.10
        ).mean()

        median_vol = vol.median()

    else:
        p_vol_3 = np.nan
        p_vol_10 = np.nan
        median_vol = np.nan

    # --------------------------------------------------------
    # Evidence
    # --------------------------------------------------------

    evidence = evidence_strength(
        n_events,
        n_dates,
        n_locations
    )

    conv_risk = conversion_risk_band(
        p_conv_2
    )

    volume_risk = volume_risk_band(
        p_vol_3,
        p_vol_10
    )

    readiness = scenario_readiness(
        evidence,
        rpu_low,
        rpu_high
    )

    scenario_rows.append({
        "Fee_Increase": fee,

        "Expected_RPU_Gain": rpu_point,
        "RPU_CI_Low_95": rpu_low,
        "RPU_CI_High_95": rpu_high,

        "Median_DiD_Conversion_Change_pp": median_conv,

        "P_Conversion_Decline_GT_1pp": p_conv_1,
        "P_Conversion_Decline_GT_2pp": p_conv_2,
        "P_Conversion_Decline_GT_5pp": p_conv_5,

        "Conversion_Risk": conv_risk,

        "Median_DiD_Volume_Change_Pct": median_vol,

        "P_Volume_Decline_GT_3pct": p_vol_3,
        "P_Volume_Decline_GT_10pct": p_vol_10,

        "Volume_Risk": volume_risk,

        "Historical_Events": n_events,
        "Unique_Locations": n_locations,
        "Corporate_Change_Dates": n_dates,

        "Evidence_Strength": evidence,
        "Readiness": readiness,
    })

decision = pd.DataFrame(
    scenario_rows
)

display(decision)


## 4. Add interpretation fields

In [ ]:

def ci_interpretation(row):

    lo = row["RPU_CI_Low_95"]
    hi = row["RPU_CI_High_95"]

    if pd.isna(lo) or pd.isna(hi):
        return "Insufficient independent dates for stable CI"

    if lo > 0:
        return "RPU gain consistently positive in grouped bootstrap"

    if hi < 0:
        return "RPU response consistently negative"

    return "RPU confidence interval includes zero"


def scenario_note(row):

    if row["Evidence_Strength"] == "Limited":
        return (
            "Limited direct historical support; "
            "use for research/scenario planning only."
        )

    if (
        row["Conversion_Risk"] == "High"
        or row["Volume_Risk"] == "High"
    ):
        return (
            "Material downside risk observed historically; "
            "review segment-level detail before action."
        )

    if row["Evidence_Strength"] == "High":
        return (
            "Strong historical support for decision-support review."
        )

    return (
        "Moderate historical support; "
        "review alongside local business context."
    )


decision["RPU_CI_Interpretation"] = decision.apply(
    ci_interpretation,
    axis=1
)

decision["Scenario_Note"] = decision.apply(
    scenario_note,
    axis=1
)

display(
    decision[
        [
            "Fee_Increase",
            "Expected_RPU_Gain",
            "RPU_CI_Low_95",
            "RPU_CI_High_95",
            "Conversion_Risk",
            "Volume_Risk",
            "Evidence_Strength",
            "Readiness",
            "Scenario_Note"
        ]
    ]
)


## 5. Save final outputs

In [ ]:

decision.to_csv(
    OUTPUT_DIR / "01_V9_1_DECISION_TABLE.csv",
    index=False
)

executive = decision[
    [
        "Fee_Increase",

        "Expected_RPU_Gain",
        "RPU_CI_Low_95",
        "RPU_CI_High_95",
        "RPU_CI_Interpretation",

        "P_Conversion_Decline_GT_1pp",
        "P_Conversion_Decline_GT_2pp",
        "P_Conversion_Decline_GT_5pp",
        "Conversion_Risk",

        "P_Volume_Decline_GT_3pct",
        "P_Volume_Decline_GT_10pct",
        "Volume_Risk",

        "Historical_Events",
        "Unique_Locations",
        "Corporate_Change_Dates",

        "Evidence_Strength",
        "Readiness",
        "Scenario_Note",
    ]
].copy()

executive.to_csv(
    OUTPUT_DIR / "02_V9_1_EXECUTIVE_SCENARIO_OUTPUT.csv",
    index=False
)

print("=" * 80)
print("V9.1 COMPLETE")
print("=" * 80)

print("\nOutput folder:")
print(OUTPUT_DIR)

print("\nFiles:")
print("01_V9_1_DECISION_TABLE.csv")
print("02_V9_1_EXECUTIVE_SCENARIO_OUTPUT.csv")

display(executive)


## 6. Additional stability diagnostics

In [ ]:

stability_rows = []

for fee in SCENARIOS:

    d = did[
        did["BUYFEE_CHANGE_AMOUNT"].between(
            fee - 2.5,
            fee + 2.5
        )
    ].copy()

    if d.empty:
        continue

    per_date = (
        d.groupby("Corporate_Change_ID")
        .agg(
            Events=("Segment_Change_ID", "count"),
            Median_RPU=("DiD_RPU_Change_Dollar", "median"),
            Median_Conversion=("DiD_Conversion_Change_pp", "median"),
            Median_Volume=("DiD_Volume_Change_Pct", "median"),
        )
        .reset_index()
    )

    stability_rows.append({
        "Fee_Increase": fee,
        "Corporate_Change_Dates": len(per_date),

        "Pct_Dates_RPU_Positive": (
            per_date["Median_RPU"] > 0
        ).mean(),

        "Pct_Dates_Conversion_NonNegative": (
            per_date["Median_Conversion"] >= 0
        ).mean(),

        "Pct_Dates_Volume_NonNegative": (
            per_date["Median_Volume"] >= 0
        ).mean(),

        "Median_Date_RPU": per_date["Median_RPU"].median(),
        "Median_Date_Conversion": per_date["Median_Conversion"].median(),
        "Median_Date_Volume": per_date["Median_Volume"].median(),
    })

stability = pd.DataFrame(
    stability_rows
)

stability.to_csv(
    OUTPUT_DIR / "03_V9_1_DATE_STABILITY.csv",
    index=False
)

display(stability)



## What to share back

Please upload:

1. `02_V9_1_EXECUTIVE_SCENARIO_OUTPUT.csv`
2. `03_V9_1_DATE_STABILITY.csv`
3. `01_V9_1_DECISION_TABLE.csv`

These three files are enough to decide whether the scenario evidence is strong enough for an Excel pricing decision tool.

## Why this version should run much more reliably

V9.1:
- does not load the large weekly dataset,
- does not use Random Forest,
- does not use one-hot encoding,
- does not create large feature matrices,
- works only from the already-small V8.2 event-level DiD output,
- uses simple pandas / numpy grouped calculations.
